In [ ]:
# CELL 1 and 2: Install + clone (run once per session, no kernel restart needed)
import subprocess, sys, os, torch

# Clone latest repo
if os.path.exists('/kaggle/working/thesis'):
    subprocess.run(['rm', '-rf', '/kaggle/working/thesis'], check=True)
subprocess.run([
    'git', 'clone',
    'https://github.com/taratorbati/thesis.git',
    '/kaggle/working/thesis'
], check=True)

os.chdir('/kaggle/working/thesis')
sys.path.insert(0, '/kaggle/working/thesis')

# Install deps
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'stable-baselines3==2.6.0',
    'gymnasium==1.1.1',
    'casadi',
], check=True)

import numpy as np, gymnasium, stable_baselines3 as sb3
print(f'numpy:             {np.__version__}')
print(f'gymnasium:         {gymnasium.__version__}')
print(f'stable-baselines3: {sb3.__version__}')
print(f'PyTorch:           {torch.__version__}')
print(f'CUDA:              {torch.cuda.is_available()}')
torch.set_num_threads(4)
print('Setup complete.')

In [ ]:
# CELL 3: Smoke test (~2 min) — READ OUTPUT before running Cell 4
import time, numpy as np
from stable_baselines3 import SAC
from src.rl.gym_env import IrrigationEnv
from src.rl.networks import CTDESACPolicy, make_sac_policy_kwargs

# IrrigationEnv() with no fixed_scenario: training mode (random year/budget)
env = IrrigationEnv(seed=0)
obs, _ = env.reset()
print(f'obs_dim:    {obs.shape[0]} (expected 707)')
print(f'action_dim: {env.action_space.shape[0]} (expected 130)')

model = SAC(
    policy=CTDESACPolicy, env=env,
    policy_kwargs=make_sac_policy_kwargs(N=130),
    buffer_size=10_000, batch_size=256,
    learning_starts=10, gradient_steps=1, verbose=0,
)
model.learn(total_timesteps=50, reset_num_timesteps=True)
t0 = time.time()
model.learn(total_timesteps=200, reset_num_timesteps=False)
rate = 200 / (time.time() - t0)
print(f'Device: {model.device}')
print(f'Rate:   {rate:.1f} steps/sec  ->  est {500_000/rate/3600:.1f} hrs for 500k steps')
print('PASSED' if obs.shape[0] == 707 else 'FAILED: wrong obs_dim')

In [ ]:
# CELL 4: Training
# Change SEED for each of the 5 parallel notebooks (0, 1, 2, 3, 4)
from src.rl.train import train_sac

SEED = 0  # use 0,1,2,3,4 across 5 parallel notebooks

train_sac(
    total_timesteps=500_000,
    seed=SEED,
    output_dir='/kaggle/working/thesis/results/rl',
    checkpoint_freq=50_000,
    eval_freq=25_000,
    verbose=1,
)

In [ ]:
# CELL 5: Save outputs — run after Cell 4 completes
import shutil, os

src = '/kaggle/working/thesis/results/rl'
dst = '/kaggle/working/results_rl'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)

for d in sorted(os.listdir(dst)):
    n = sum(len(fs) for _, _, fs in os.walk(os.path.join(dst, d)))
    print(f'  {d}  ({n} files)')
print('Download from the Output tab.')

In [ ]:
# CELL 6: Resume interrupted run — DO NOT RUN on a fresh session
# 1. Upload previous results_rl as a Kaggle Dataset
# 2. Add it as Input to this notebook  
# 3. Update CHECKPOINT_PATH below
# 4. Run Cell 2, then this cell, then Cell 5
import shutil, os
from src.rl.train import train_sac

PREV_RESULTS    = '/kaggle/input/thesis-rl-checkpoints/results_rl'
CHECKPOINT_PATH = 'results/rl/sac_general_seed0/checkpoints/sac_general_seed0_300000_steps.zip'
SEED = 0

dst = '/kaggle/working/thesis/results/rl'
os.makedirs(dst, exist_ok=True)
shutil.copytree(PREV_RESULTS, dst, dirs_exist_ok=True)
print('Checkpoints restored.')

train_sac(
    total_timesteps=500_000, seed=SEED,
    output_dir='/kaggle/working/thesis/results/rl',
    checkpoint_freq=50_000, eval_freq=25_000,
    resume_path=CHECKPOINT_PATH, verbose=1,
)